# 06 — Stage 2: Conventional and Corrected Alpha (Δα̂)

Both legs use the seven-factor model (FF5 + MOM + Pástor–Stambaugh traded liquidity), fund-by-fund OLS with
Newey–West(12) standard errors:

- **Stage 2a (conventional):** R<sub>i,t</sub> − r<sub>f,t</sub> on the factors → α̂<sub>conv</sub>
- **Stage 2b (corrected):** R̃<sub>i,t</sub> − r<sub>f,t</sub> on the factors → α̂<sub>corr</sub>
- **Estimand:** Δα̂ = α̂<sub>corr</sub> − α̂<sub>conv</sub>

**Net returns.** R<sub>i,t</sub> is the CRSP return net of expense ratios and 12b-1 fees (loads not deducted), so α̂<sub>conv</sub>,
α̂<sub>corr</sub> and Δα̂ are net-of-fee alphas: what an investor earned after costs. Judging manager *skill* would favour
gross returns (net return + expense ratio / 12, cf. Fama and French 2010, who report both); this is not implemented
and is noted as a limitation.

**Common fund set.** Both legs are estimated on identical rows: a fund-month is used only if the raw return, the
corrected return and all seven factors are present, and a fund needs ≥ 60 such months. Δα̂ is therefore a within-fund
difference, not a change in sample composition. The dependent variable is not winsorised (winsorisation is a
robustness check in NB07).

| | |
|---|---|
| **Input** | `stage1b_corrected_returns.parquet` (NB05) |
| **Outputs** | `stage2a_common_results.parquet`, `stage2b_results.parquet`, `stage2a_common_residuals.parquet`, `stage2b_residuals.parquet`, `delta_alpha.parquet` → read by NB07, NB08 |
| **Thesis** | Sections 4.4, 5.1, 5.3 (headline), Figure 3 |
| **Expected** | 1,950 funds, 318,863 fund-months; Δα̂ mean −0.0173%/yr, t = −2.76 |

**To replicate:** set `DATA_ROOT` in `thesis_config.py`, run NB01–NB05, then *Restart Kernel → Run All*.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from thesis_config import (DATA_DIR, FACTORS, MIN_OBS, assert_matches, check_paths, estimate_alpha, save_fig,
                           save_parquet)

check_paths(DATA_DIR / "stage1b_corrected_returns.parquet")
print(f"pandas {pd.__version__} | numpy {np.__version__}")

## 1. Common fund set

In [ ]:
df = pd.read_parquet(DATA_DIR / "stage1b_corrected_returns.parquet")
missing = [c for c in FACTORS + ["excess_ret", "excess_r_corrected", "wficn", "caldt"] if c not in df.columns]
assert not missing, f"Missing columns: {missing}"
print(f"Stage 1b panel: {df.shape}, {df['wficn'].nunique()} funds")

n_overlap    = df.dropna(subset=FACTORS + ["excess_ret", "excess_r_corrected"]).groupby("wficn").size()
common_funds = sorted(n_overlap[n_overlap >= MIN_OBS].index)

valid_raw  = df.dropna(subset=FACTORS + ["excess_ret"]).groupby("wficn").size()
valid_corr = df.dropna(subset=FACTORS + ["excess_r_corrected"]).groupby("wficn").size()
funds_raw  = set(valid_raw[valid_raw >= MIN_OBS].index)
funds_corr = set(valid_corr[valid_corr >= MIN_OBS].index)
dropped    = funds_raw - funds_corr
print(f"Funds with ≥{MIN_OBS} raw months: {len(funds_raw)} | corrected: {len(funds_corr)} | common: {len(common_funds)}")

# Why funds drop out: too few correctable months, almost all of them pre-2003 (per_cash coverage)
corr_counts = valid_corr.reindex(sorted(dropped)).fillna(0).astype(int)
nan_corr = df[df["wficn"].isin(dropped) & df["excess_r_corrected"].isna()]
print(f"Dropped {len(dropped)}: {(corr_counts == 0).sum()} with no correctable month, "
      f"{((corr_counts > 0) & (corr_counts < MIN_OBS)).sum()} with 1–{MIN_OBS-1}")
print(f"  their uncorrectable months: per_cash missing {nan_corr['per_cash_clean'].isna().mean():.1%}, "
      f"benchmark missing {nan_corr['r_passive'].isna().mean():.1%}; "
      f"share before 2003: {(nan_corr['caldt'].dt.year < 2003).mean():.1%}")

assert_matches("Section 4.4: funds (all, common, dropped)", [len(funds_raw), len(common_funds), len(dropped)],
               [2_138, 1_950, 188], 0)

## 2. Estimation

In [ ]:
results_2b, residuals_2b = estimate_alpha(df, common_funds, "excess_r_corrected", "2b corrected")
results_2a, residuals_2a = estimate_alpha(df, common_funds, "excess_ret",         "2a conventional")
assert set(results_2a["wficn"]) == set(results_2b["wficn"]), "Fund sets differ between legs"

delta = (results_2a[["wficn", "alpha", "alpha_tstat", "alpha_pval"]]
         .rename(columns={"alpha": "alpha_conv", "alpha_tstat": "tstat_conv", "alpha_pval": "pval_conv"})
         .merge(results_2b[["wficn", "alpha", "alpha_tstat", "alpha_pval"]]
                .rename(columns={"alpha": "alpha_corr", "alpha_tstat": "tstat_corr", "alpha_pval": "pval_corr"}),
                on="wficn", how="inner"))
delta["delta_alpha"] = delta["alpha_corr"] - delta["alpha_conv"]
print(f"Δα̂ for {len(delta)} funds, {len(residuals_2b):,} fund-months")

save_parquet(results_2a,   DATA_DIR / "stage2a_common_results.parquet",   index=False)
save_parquet(results_2b,   DATA_DIR / "stage2b_results.parquet",          index=False)
save_parquet(residuals_2a, DATA_DIR / "stage2a_common_residuals.parquet", index=False)
save_parquet(residuals_2b, DATA_DIR / "stage2b_residuals.parquet",        index=False)
save_parquet(delta,        DATA_DIR / "delta_alpha.parquet",              index=False)

## 3. Section 5.1 — conventional alpha

In [ ]:
a = results_2a["alpha"] * 1200
n = len(results_2a)
pos     = (results_2a["alpha"] > 0).sum()
sig_pos = ((results_2a["alpha"] > 0) & (results_2a["alpha_pval"] < 0.05)).sum()
sig_neg = ((results_2a["alpha"] < 0) & (results_2a["alpha_pval"] < 0.05)).sum()
print(f"Conventional α (%/yr): mean {a.mean():+.3f}, median {a.median():+.3f}")
print(f"Positive: {pos} ({pos/n:.1%}) | significant positive: {sig_pos} ({sig_pos/n:.1%}) | "
      f"significant negative: {sig_neg} ({sig_neg/n:.1%})  [p < 0.05]")

b = results_2b.merge(results_2a, on="wficn", suffixes=("_corr", "_conv"))
print(f"Median β_mkt: conventional {b['beta_mkt_conv'].median():.3f} → corrected {b['beta_mkt_corr'].median():.3f}")

assert_matches("Section 5.1: conventional α mean, median (%/yr)", [a.mean(), a.median()], [-0.937, -1.047], 3)
assert_matches("Section 5.1: positive, sig. positive, sig. negative", [pos, sig_pos, sig_neg], [513, 58, 438], 0)
assert_matches("Section 5.1/5.3: median β_mkt conv → corr", [b["beta_mkt_conv"].median(), b["beta_mkt_corr"].median()],
               [0.988, 1.008], 3)

## 4. Section 5.3 — headline Δα̂

In [ ]:
d = delta["delta_alpha"].dropna()
t_stat, p_val = stats.ttest_1samp(d, 0.0)
n_neg = (d < 0).sum()
sign_p = stats.binomtest(n_neg, len(d), p=0.5).pvalue
w_stat, w_p = stats.wilcoxon(d)
ci = ((d.mean() - 1.96 * d.sem()) * 1200, (d.mean() + 1.96 * d.sem()) * 1200)

for name, col in [("Conventional α", "alpha_conv"), ("Corrected α   ", "alpha_corr"), ("Δα̂           ", "delta_alpha")]:
    s = delta[col] * 1200
    print(f"{name} (%/yr): mean {s.mean():+.4f}, median {s.median():+.4f}")
print(f"\nt = {t_stat:.2f}, p = {p_val:.4f} | 95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}] %/yr")
print(f"Sign test: {n_neg}/{len(d)} negative ({n_neg/len(d):.1%}), p = {sign_p:.4f} | Wilcoxon p = {w_p:.4f}")
print(f"SD of Δα̂ across funds: {d.std()*1200:.4f}%/yr")

assert_matches("Section 5.3: Δα̂ mean (%/yr)", d.mean() * 1200, -0.0173, 4)
assert_matches("Section 5.3: t-statistic", t_stat, -2.76, 2)
assert_matches("Section 5.3: 95% CI (%/yr)", ci, [-0.030, -0.005], 3)
assert_matches("Section 5.3: corrected α mean, median (%/yr)",
               [(delta["alpha_corr"] * 1200).mean(), (delta["alpha_corr"] * 1200).median()], [-0.954, -1.050], 3)

**Figure 3 — cross-sectional distribution of Δα̂.**

In [ ]:
delta_ann = delta["delta_alpha"] * 1200
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(delta_ann.clip(-2, 2), bins=80, color="#2c7bb6", edgecolor="none", alpha=0.85)
ax.axvline(0, color="black", linewidth=1)
ax.axvline(delta_ann.mean(),   color="#d7191c", linewidth=1.5, linestyle="--", label=f"Mean: {delta_ann.mean():+.2f}%")
ax.axvline(delta_ann.median(), color="#1a9641", linewidth=1.5, linestyle="--", label=f"Median: {delta_ann.median():+.2f}%")
ax.set_xlabel("Δα̂ (annualised %, clipped ±2% for display)", fontsize=12)
ax.set_ylabel("Number of Funds", fontsize=12)
ax.set_title("Distribution of the Friction Correction Δα̂ = α_corr − α_conv", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
save_fig(fig, "fig3_delta_alpha_distribution")
plt.show()